# Dask support
Like xarray, xarray-einstats also aims to support using their functions on
NumPy or Dask backed xarray objects.

Functions in xarray-einstats should support Dask backed xarray objects.
But Dask support is still not extensively tested.
If you have issues using Dask backed xarray objects please open an [issue](https://github.com/arviz-devs/xarray-einstats)

Dask support within xarray-einstats requires explicit activation by users, exactly like it happens with {func}`xarray.apply_ufunc`.
In fact, what is happening is xarray-einstats forwards any unrecognized keyword argument (e.g. `dask` or `dask_gufunc_kwargs`) to `apply_ufunc`.

In [1]:
import dask.array as da
import xarray as xr
from xarray_einstats import linalg

xr.set_options(display_expand_data=False);

In [2]:
matrix_batch = xr.DataArray(da.random.normal(size=(1000, 77, 77), chunks="auto"), dims=["batch", "dim1", "dim2"])

Trying to use xarray-einstats functions directly on Dask arrays triggers an error:

In [3]:
linalg.norm(matrix_batch, dims=["dim1", "dim2"])

ValueError: apply_ufunc encountered a chunked array on an argument, but handling for chunked arrays has not been enabled. Either set the ``dask`` argument or load your data into memory first with ``.load()`` or ``.compute()``

We need to allow dask inputs through `dask="allowed"` or `dask="parallelized"`. In general, `allowed` is preferred, but it only works if there is a Dask implementation of the requested function available, `parallelized` works in most cases but is usually slower and still not a silver bullet. Consequently, xarray-einstats does not set the `dask` argument automatically when the input is a dask array, and finding which of the two will work and be better might require a bit of trial an error an also be dependent on the Dask version available.

Here is one example with `linalg.norm` for which both alternatives work:

In [4]:
norms = linalg.norm(matrix_batch, dims=["dim1", "dim2"], dask="allowed")
norms.data

dask.array<pow, shape=(1000,), dtype=float64, chunksize=(1000,), chunktype=numpy.ndarray>

In [5]:
norms.compute()

<xarray.DataArray 'normal-758a57bb75ed3127c94212c3e83a5ed8' (batch: 1000)> Size: 8kB
77.07 77.61 76.86 77.53 77.99 77.22 76.9 ... 77.92 78.03 77.96 76.6 77.31 76.26
Dimensions without coordinates: batch

In [6]:
linalg.norm(matrix_batch, dims=["dim1", "dim2"], dask="parallelized").compute()

<xarray.DataArray 'normal-758a57bb75ed3127c94212c3e83a5ed8' (batch: 1000)> Size: 8kB
77.07 77.61 76.86 77.53 77.99 77.22 76.9 ... 77.92 78.03 77.96 76.6 77.31 76.26
Dimensions without coordinates: batch

:::{seealso}
{ref}`Xarray documentation <xarray:dask>`

[Dask examples](https://examples.dask.org/xarray.html)
:::

In [7]:
%load_ext watermark
%watermark -n -u -v -iv -w

Last updated: Mon, 20 Jul 2026

Python implementation: CPython
Python version       : 3.14.6
IPython version      : 9.15.0

dask           : 2026.7.1
xarray         : 2026.7.0
xarray_einstats: 0.11.0

Watermark: 2.6.0

